PROJET MODELISATION GEOMETRIQUE

MODELISATION D'UNe INVASION ZOMBIE

Partie 1 : Mouvement Aléatoire

In [17]:
#IMPORTS 

import random
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

import plotly.graph_objects as go
import plotly.express as px


In [28]:
#Initialisation des explorateurs

explorateurs = []
nb_explorateurs = 20
zone_limite = 100
precision = 1000


#Les explorateurs sont initialisés avec des positions et des vitesses aléatoires dans un espace de taille définie par zone_limite. Les vitesses sont également aléatoires, ce qui permet de simuler un mouvement naturel des explorateurs dans le vaisseau (sans gravité).
#Chaque explorateur est représenté par un identifiant unique (i), une position (x, y, z) et une vitesse (vx, vy, vz). Le paramètre False à la fin de la ligne d'ajout de l'explorateur peut être utilisé pour indiquer si l'explorateur a été contaminé par un zombie ou non.
for i in range(nb_explorateurs):
    x = zone_limite*random.randint(0, precision)/precision
    y = zone_limite*random.randint(0, precision)/precision
    z = zone_limite*random.randint(0, precision)/precision
    vx = zone_limite*random.randint(-precision, precision)/precision
    vy = zone_limite*random.randint(-precision, precision)/precision
    vz = zone_limite*random.randint(-precision, precision)/precision
    explorateurs.append([i, np.array([x, y, z]), np.array([vx, vy, vz]), False])

#Génération aléatoire d'un infecté parmi les explorateurs
def contaminer_explorateur(explorateurs):
    index = random.randint(0, len(explorateurs) - 1)
    explorateurs[index][3] = True
    
#On infecte un explorateur
contaminer_explorateur(explorateurs)




In [20]:
#Fonction de mise à jour des positions et des vitesses des explorateurs

def update_explorateurs(explorateurs, dt):
    for i in range(len(explorateurs)):
        _, pos, vel, _ = explorateurs[i]
        # Mise à jour de la position en fonction de la vitesse
        pos += vel * dt
        for j in range(3):
            if pos[j] < -zone_limite or pos[j] > zone_limite:
                vel[j] = -vel[j]  # Inversion de la vitesse pour rebondir sur les limites

In [42]:
#Première modélisation 3D avec Plotly

frames_data = []
dt = 0.1  
nb_frames = 200
for frame in range(nb_frames):
    update_explorateurs(explorateurs, dt)
    x = [e[1][0] for e in explorateurs]
    y = [e[1][1] for e in explorateurs]
    z = [e[1][2] for e in explorateurs]
    infect = [e[3] for e in explorateurs]
    frames_data.append((x, y, z, infect))

x_cage = [-1, 1, 1, -1, -1, -1, 1, 1, -1, -1, 1, 1, 1, 1, -1, -1]
y_cage = [-1, -1, 1, 1, -1, -1, -1, 1, 1, -1, -1, -1, 1, 1, 1, 1]
z_cage = [-1, -1, -1, -1, -1, 1, 1, 1, 1, 1, 1, -1, -1, 1, 1, -1]

x_cage = [x * zone_limite for x in x_cage]
y_cage = [y * zone_limite for y in y_cage]
z_cage = [z * zone_limite for z in z_cage]


couleurs_initiales = ['red' if est_infecte else 'blue' for est_infecte in frames_data[0][3]]
symbol_initiaux=['cross' if est_infecte else 'circle' for est_infecte in frames_data[0][3]]

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=x_cage, y=y_cage, z=z_cage, 
            mode='lines', 
            line=dict(color='cyan', width=4)),

        go.Scatter3d(
            x=frames_data[0][0], 
            y=frames_data[0][1], 
            z=frames_data[0][2], 
            mode='markers', 
            marker=dict(size=5, color=couleurs_initiales, symbol=symbol_initiaux))]
)

#Pour chaque frame, on met à jour les positions et les couleurs des points en fonction de l'état d'infection des explorateurs.
frames_tempo = []

for frame in frames_data:
    couleurs = ['red' if est_infecte else 'blue' for est_infecte in frame[3]]
    symbol = ['cross' if est_infecte else 'circle' for est_infecte in frame[3]]
    frames_tempo.append(
    go.Frame(data=[go.Scatter3d(x=x_cage, y=y_cage, z=z_cage, mode='lines', line=dict(color='cyan', width=4)),
                    go.Scatter3d(x=frame[0], y=frame[1], z=frame[2], marker=dict(size=5, color=couleurs, symbol=symbol))
                   ])
)

fig.frames = frames_tempo

no_axis = dict(
    showbackground=False,
    showticklabels=False,
    showgrid=False,
    showline=False,
    zeroline=False
)

fig.update_layout(
    title='Simulation des explorateurs flottants dans le vaisseau',
    paper_bgcolor='black',
    scene=dict(
        xaxis=dict(**no_axis, range=[-zone_limite, zone_limite]),
        yaxis=dict(**no_axis, range=[-zone_limite, zone_limite]),
        zaxis=dict(**no_axis, range=[-zone_limite, zone_limite]),
        aspectmode='cube'
    ),
    updatemenus=[dict(
        type='buttons',
        buttons=[dict(label='Play', method='animate', args=[None, {'frame': {'duration': 50, 'redraw': True}, 'fromcurrent': True}])]
    )]
)

fig.show()
    